# 00. Data Audit & Core Cleaning Pipeline
### Full-Cycle Data Project (Shared Core)

**Deskripsi Notebook:**
Notebook ini merupakan gerbang utama (*Single Source of Truth*) untuk memeriksa integritas data mentah (*Data Quality Audit*) dan melakukan standarisasi pembersihan sebelum data didistribusikan ke **Track Analytics** dan **Track Science**.

**Aturan Kerja:**
- Pembersihan data hanya dilakukan satu kali di modul `src/telco_churn/core/cleaner.py`.
- Semua track selanjutnya wajib membaca data bersih dari `data/processed/telco_churn_clean.csv`.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Pastikan root direktori terdaftar dalam sys.path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.telco_churn.core.loader import load_raw_data, load_processed_data, PROCESSED_DIR
from src.telco_churn.core.cleaner import clean_column_names, clean_telco_data, save_processed_data

print("Project Root:", PROJECT_ROOT)


## 1. Memuat Data Mentah (*Raw Data*)
Data mentah dibaca secara *read-only* dari direktori `data/raw/` tanpa modifikasi langsung pada file sumber.


In [ ]:
raw_df = load_raw_data()
print(f"Dimensi Data Mentah: {raw_df.shape[0]:,} baris x {raw_df.shape[1]} kolom")
raw_df.head()


## 2. Audit Kualitas Data (*Data Quality Audit*)
Pemeriksaan mencakup:
1. Pengecekan nilai duplikat pada primary key (`customerID`).
2. Pengecekan tipe data masing-masing kolom.
3. Deteksi missing values tersembunyi (misal string spasi kosong pada `TotalCharges`).


In [ ]:
# 1. Pengecekan Duplikasi Customer ID
dup_ids = raw_df["customerID"].duplicated().sum()
print(f"Jumlah Customer ID Duplikat: {dup_ids}")

# 2. Ringkasan Tipe Data
print("\nInfo Tipe Data:")
print(raw_df.dtypes.value_counts())


In [ ]:
# 3. Deteksi Whitespace Anomali pada TotalCharges
whitespace_mask = raw_df["TotalCharges"].str.strip() == ""
whitespace_count = whitespace_mask.sum()
print(f"Jumlah baris dengan spasi kosong pada TotalCharges: {whitespace_count}")

# Analisis konteks bisnis: apakah ini pelanggan baru?
anomalous_rows = raw_df[whitespace_mask]
print("Distribusi tenure pelanggan dengan TotalCharges kosong:")
print(anomalous_rows["tenure"].value_counts())


> **Temuan Audit & Kebijakan Bisnis:**
> Seluruh pelanggan yang memiliki spasi kosong pada `TotalCharges` (11 baris) memiliki `tenure = 0`. Ini berarti mereka adalah pelanggan baru yang belum menyelesaikan siklus penagihan pertama. 
> Sesuai logika bisnis telekomunikasi, nilai `TotalCharges` yang tepat adalah `0.0`, bukan menghapus baris ataupun imputasi rata-rata/median.


## 3. Eksekusi Standarisasi & Pembersihan Data
Menerapkan fungsi `clean_telco_data()` dari paket inti `src.telco_churn.core.cleaner`:
1. Standarisasi nama kolom ke format `snake_case` (contoh: `customerID` -> `customer_id`, `MonthlyCharges` -> `monthly_charges`).
2. Konversi kolom `total_charges` ke float numerik dan imputasi anomali tenure 0 dengan `0.0`.


In [ ]:
clean_df = clean_telco_data(raw_df)
print("Dimensi Data Bersih:", clean_df.shape)
print("Kolom yang telah distandarisasi:")
print(clean_df.columns.tolist()[:8], "...")


In [ ]:
# Verifikasi Pasca Pembersihan
print(f"Total Missing Values di Seluruh Kolom: {clean_df.isna().sum().sum()}")
print(f"Tipe Data total_charges: {clean_df['total_charges'].dtype}")
print(f"Nilai minimum total_charges untuk tenure=0: {clean_df[clean_df['tenure'] == 0]['total_charges'].unique()}")


## 4. Ekspor ke Single Source of Truth
Menyimpan dataset bersih ke `data/processed/telco_churn_clean.csv`.


In [ ]:
saved_path = save_processed_data(clean_df)
print(f"Dataset bersih berhasil disimpan ke: {saved_path}")


## Ringkasan Audit
- Dataset bersih siap digunakan untuk **Track Analytics** (`notebooks/01_analytics/`) dan **Track Science** (`notebooks/02_prediction/`).
- Integritas data terjamin: 7,043 baris, 21 kolom, 0 missing value, 0 duplikat customer id.
